<a href="https://colab.research.google.com/github/Not-kh-lily-23/dbank-longitudinal-prediction/blob/main/feature_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

first code failed miserably :(((((((( why doesnt this allow emojis???

In [2]:
!pip install pylangacq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.0 MB/s eta 0:00:00


In [4]:
import pandas as pd
import pylangacq
import os
drive.mount('/content/drive')
def calc_mattr(words,window_size=50):
    if len(words)<window_size:
        return len(set(words))/len(words) if words else 0
    ttr_values=[]
    for i in range(len(words)-window_size+1):
        window=words[i:i+window_size]
        ttr_values.append(len(set(window))/window_size)
    return sum(ttr_values)/len(ttr_values)
base='/content/drive/MyDrive/DementiaBank Project'
mcsv=os.path.join(base,'cross_sectional_baseline_master.csv')
df=pd.read_csv(mcsv)
df=df[df['cohort_status'].isin(['Stable Control','Stable MCI','Stable AD'])].copy()
print(f"total usable files: {len(df)}")
def ext_ling_feat(filepath):
    try:
        chat=pylangacq.read_chat(filepath)
        par_utterances=[utt for utt in chat.utterances() if utt.participant=='PAR']
        if not par_utterances or len(par_utterances)==0:
            return None
        tokens=[token for utt in par_utterances for token in utt.tokens]
        words=[t.word for t in tokens if t.word]
        clean=[w.lower() for w in words if w.isalpha()]
        count=len(clean)
        if count==0:
            return None
        mlu_w=len(words)/len(par_utterances)
        mattr=calc_mattr(clean)
        noun=0
        pron=0
        for token in tokens:
            pos=str(token.pos).lower()
            if pos.startswith('n') and not pos.startswith('num'):
                noun+=1
            elif pos.startswith('pro'):
                pron+=1
        pn_ratio=pron/(noun+pron) if (noun+pron)>0 else 0
        raw_text=" ".join([utt.tiers[utt.participant] for utt in par_utterances])
        filled_pauses=raw_text.count('&-')
        retraces=raw_text.count('[/]')
        corrections=raw_text.count('[//]')
        stutters=raw_text.count('&~')
        total_disfluencies=filled_pauses+retraces+corrections+stutters
        dfl_rate=(total_disfluencies/count)*100 if count>0 else 0
        return {
            'word_count': count,
            'mlu_w': mlu_w,
            'mattr': mattr,
            'pn_ratio': pn_ratio,
            'disfluency_rate': dfl_rate
        }
    except Exception as e:
        print(f"Failed to process {filepath}: {e}")
        return None
f_list=[]
for index,row in df.iterrows():
    fpath=row['file_path']
    if not os.path.exists(fpath):
        continue
    feats=ext_ling_feat(fpath)
    if feats:
        combined_data={**row.to_dict(),**feats}
        f_list.append(combined_data)
feat_df=pd.DataFrame(f_list)
op=os.path.join(base,'nlp_features_final.csv')
feat_df.to_csv(op,index=False)
print(f"Saved {len(feat_df)} rows to {op}")
display(feat_df[['participant_id','cohort_status','mlu_w','mattr','pn_ratio','disfluency_rate']].head(5))

Mounted at /content/drive
total usable files: 281
Saved 281 rows to /content/drive/MyDrive/DementiaBank Project/nlp_features_final.csv


,participant_id,cohort_status,mlu_w,mattr,pn_ratio,disfluency_rate
0,1,Stable AD,8.600000,0.624762,0.304348,15.714286
1,2,Stable Control,8.388889,0.743733,0.244444,6.451613
2,3,Stable AD,9.304348,0.712030,0.473684,9.890110
3,5,Stable AD,6.666667,0.680851,0.266667,12.765957
4,6,Stable Control,7.357143,0.684375,0.333333,7.407407
